# Customer support ticket router

Run the cells in order in Google Colab. Tickets containing billing keywords go to Billing. All other tickets go to Technical. The router requires no API key.

In [ ]:
%pip install -q langgraph

In [ ]:
import re
from typing import Literal, NotRequired, TypedDict

from langgraph.graph import END, START, StateGraph

Category = Literal["billing", "technical"]
BILLING_WORDS = {"bill", "billing", "charge", "charged", "invoice", "payment", "refund"}


class TicketState(TypedDict):
    ticket: str
    category: NotRequired[Category]
    reply: NotRequired[str]


def classify(state: TicketState) -> dict[str, Category]:
    words = set(re.findall(r"\b\w+\b", state["ticket"].casefold()))
    category: Category = "billing" if words & BILLING_WORDS else "technical"
    return {"category": category}


def route_ticket(state: TicketState) -> Category:
    return state["category"]


def billing(state: TicketState) -> dict[str, str]:
    return {"reply": "Please share your invoice number so we can review the charge."}


def technical(state: TicketState) -> dict[str, str]:
    return {"reply": "Please share the error message and the steps that triggered it."}


builder = StateGraph(TicketState)
builder.add_node("classify", classify)
builder.add_node("billing", billing)
builder.add_node("technical", technical)
builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify", route_ticket, {"billing": "billing", "technical": "technical"}
)
builder.add_edge("billing", END)
builder.add_edge("technical", END)
router = builder.compile()

In [ ]:
for ticket in ("I was charged twice this month.", "The app crashes when I sign in."):
    result = router.invoke({"ticket": ticket})
    print(f"Ticket: {result['ticket']}")
    print(f"Category: {result['category']}")
    print(f"Reply: {result['reply']}")
    print()

In [ ]:
from IPython.display import Image, display

display(Image(router.get_graph().draw_mermaid_png()))